In [0]:
from pyspark.sql.functions import *
# dbutils.widgets.text("KAFKA_BOOTSTRAP", "")
# kafka_bootstrap = dbutils.widgets.get("KAFKA_BOOTSTRAP")

dbutils.widgets.text("KAFKA_BOOTSTRAP", "")
dbutils.widgets.text("TOPIC_PATTERN", "cdc.public.*")

CONFIG = {
    "kafka_bootstrap": dbutils.widgets.get("KAFKA_BOOTSTRAP"),
    "topic_pattern": dbutils.widgets.get("TOPIC_PATTERN"),
    "bronze_schema": "bronze",
    "silver_schema": "silver"
}


kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", CONFIG["kafka_bootstrap"])
    .option("subscribePattern", CONFIG["topic_pattern"])
    .load()
)

In [0]:
def write_bronze(batch_df, batch_id):

    tables = [r.table_name for r in batch_df.select("table_name").distinct().collect()]

    for table in tables:

        df = batch_df.filter(col("table_name") == table)

        target = f"workspace.{CONFIG["bronze_schema"]}.{table}"

        (
            df.write
            .format("delta")
            .mode("append")
            .saveAsTable(target)
        )

In [0]:
(
events.writeStream
.foreachBatch(write_bronze)
.option("checkpointLocation","/Volumes/workspace/checkpoints/bronze_auto")
.trigger(availableNow=True)
.start()
)

In [0]:
# df = (
#   spark.readStream
#   .format("kafka")
#   .option("kafka.bootstrap.servers","2.tcp.eu.ngrok.io:14739")
#   .option("subscribe","cdc.public.orders")
#   .load()
# )

In [0]:
# from pyspark.sql.functions import col

# events = df.selectExpr("CAST(value AS STRING)")

In [0]:
# (
#   events.writeStream
#   .format("delta")
#   .option("checkpointLocation","/Volumes/workspace/default/mnt/bronze/orders/checkpoint")
#   .trigger(availableNow=True)
#   .start("/Volumes/workspace/default/mnt/bronze/orders")
# )

In [0]:
# bronze = spark.read.format("delta").load(
# "/Volumes/workspace/default/mnt/bronze/orders"
# )

# # display(bronze)